# Time Travel: Undo Your Mistakes

Run this notebook only in your provided personal workshop account, after loading the Day 0 data. Keep **ACCOUNTADMIN** selected throughout.

**Run one cell at a time, from top to bottom. Do not use Run all.** This lesson intentionally updates patient cities and drops a table. Read each checkpoint and finish both restorations before leaving. The expected-error cell must be run separately from UNDROP.

Open this file in the `snowcamp` workspace in Snowsight. Use the notebook role/warehouse picker to confirm `ACCOUNTADMIN` and select `LAB_WH`. When connecting notebook compute, use a facilitator-approved CPU notebook service with a short idle timeout (15 minutes if offered). Do not install packages or create another trial account. Keep the same notebook session through recovery; do not restart or suspend it mid-exercise.

All records are fictional teaching data. No local Python or credentials are needed.

In [ ]:
SELECT CURRENT_ACCOUNT() AS ACCOUNT_NAME,
       CURRENT_USER() AS USER_NAME,
       CURRENT_ROLE() AS ACTIVE_ROLE,
       CURRENT_WAREHOUSE() AS QUERY_WAREHOUSE;

## 1. Check the starting data

Confirm the account is your workshop account, the role is `ACCOUNTADMIN`, and the query warehouse is `LAB_WH`. If any setting is wrong, stop and ask the facilitator; do not switch roles.

Run the next two cells. Expect 500 patients, 5,000 encounters, and 1,500 notes, then ten cities whose counts sum to 500. Save the original city counts outside the notebook output (for example, a screenshot) so you can compare them after recovery. Do not continue with a partially loaded dataset.

In [ ]:
SELECT 'PATIENTS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM DAY0_LAB.HEALTHCARE.PATIENTS
UNION ALL SELECT 'ENCOUNTERS', COUNT(*) FROM DAY0_LAB.HEALTHCARE.ENCOUNTERS
UNION ALL SELECT 'CLINICAL_NOTES', COUNT(*) FROM DAY0_LAB.HEALTHCARE.CLINICAL_NOTES;

In [ ]:
SELECT CITY, COUNT(*) AS PATIENTS
FROM DAY0_LAB.HEALTHCARE.PATIENTS
GROUP BY CITY
ORDER BY CITY;

## 2. Create the safety copy (first run only)

The next cell creates `DAY0_LAB.HEALTHCARE.PATIENTS_BEFORE_EXERCISE` without replacing anything.

**If the clone already exists, stop before the UPDATE. Do not rename, replace, or drop it.** Ask the facilitator to verify it contains the original lab data. After restoring the original data and checking the saved city counts, reuse that verified clone and repeat from the timestamp cell, not this clone cell. If it cannot be verified, use the facilitator-led exact-data recovery path in the guide.

Proceed only after clone creation succeeds or the facilitator has verified the existing clone.

In [ ]:
CREATE TABLE DAY0_LAB.HEALTHCARE.PATIENTS_BEFORE_EXERCISE
CLONE DAY0_LAB.HEALTHCARE.PATIENTS;

## 3. Save the time, then make a mistake

Run the timestamp cell **once, before** the UPDATE. It saves a SQL session variable called `before_mistake`. Keep this notebook connected to the same session until the restore is complete.

Then run the UPDATE cell. It deliberately omits a WHERE clause, so all 500 patients move to Boston. Run the city-count cell afterward to see the effect.

Do not rerun the timestamp cell while the data is wrong. If the session or variable is lost, use the recovery-only section at the end instead of capturing a new timestamp.

In [ ]:
SET before_mistake = CURRENT_TIMESTAMP();

In [ ]:
-- Intentional mistake in the fictional lab table only.
UPDATE DAY0_LAB.HEALTHCARE.PATIENTS SET CITY = 'Boston';

In [ ]:
SELECT CITY, COUNT(*) AS PATIENTS
FROM DAY0_LAB.HEALTHCARE.PATIENTS
GROUP BY CITY
ORDER BY CITY;

## 4. Look at the past

The previous result should be one row: Boston, 500. The next query reads the table at the saved timestamp; it does not change today's data. Expect the original ten cities again.

Time Travel works within the table's retention period (normally one day for these permanent lab tables). If this cell reports that `before_mistake` is undefined, stop and use the recovery-only section. Do not set a new timestamp.

In [ ]:
SELECT CITY, COUNT(*) AS PATIENTS
FROM DAY0_LAB.HEALTHCARE.PATIENTS AT(TIMESTAMP => $before_mistake)
GROUP BY CITY
ORDER BY CITY;

## 5. Restore the patient data

Run the next cell only after the historical query shows the original city counts. `INSERT OVERWRITE` replaces the current patient rows with the historical rows; it does not append duplicates. Then rerun the city-count check below. Compare it with your saved original counts before continuing.

In [ ]:
INSERT OVERWRITE INTO DAY0_LAB.HEALTHCARE.PATIENTS
SELECT * FROM DAY0_LAB.HEALTHCARE.PATIENTS AT(TIMESTAMP => $before_mistake);

In [ ]:
SELECT CITY, COUNT(*) AS PATIENTS
FROM DAY0_LAB.HEALTHCARE.PATIENTS
GROUP BY CITY
ORDER BY CITY;

## 6. Bring back a dropped table

Run the DROP cell once. Next, run the separate SELECT cell: **an object-not-found error is expected** because the table was dropped. Do not treat an earlier permission or setup error as success.

After observing that error, run the **UNDROP cell directly**, then the notes-count cell. Do not use Run all above, rerun DROP, or recreate the table. Expect 1,500 notes after UNDROP. Do not leave the lesson with this table missing.

In [ ]:
DROP TABLE DAY0_LAB.HEALTHCARE.CLINICAL_NOTES;

In [ ]:
-- EXPECTED ERROR after DROP. Next run the UNDROP cell directly.
SELECT COUNT(*) AS NOTES FROM DAY0_LAB.HEALTHCARE.CLINICAL_NOTES;

In [ ]:
UNDROP TABLE DAY0_LAB.HEALTHCARE.CLINICAL_NOTES;

In [ ]:
SELECT COUNT(*) AS NOTES FROM DAY0_LAB.HEALTHCARE.CLINICAL_NOTES;

## Core checkpoint

The patient city counts match the original ten-city distribution, and `CLINICAL_NOTES` has 1,500 rows again. If either check fails, stop and ask the facilitator before moving to Streamlit.

The next clone is optional. Skip it in a short session. A clone initially shares storage with its source; changing either can add storage. If `PATIENTS_SANDBOX` already exists, skip its cell rather than replacing it.

In [ ]:
-- OPTIONAL: run only after restoring PATIENTS. Skip if the sandbox exists.
CREATE TABLE DAY0_LAB.HEALTHCARE.PATIENTS_SANDBOX
CLONE DAY0_LAB.HEALTHCARE.PATIENTS;

## Recovery only: lost timestamp or restarted session

**Skip the next two cells on the normal path.** Run them only with the facilitator after verifying that `PATIENTS_BEFORE_EXERCISE` is the original, unchanged safety copy from before this lab's mistake. Never recapture a timestamp while the data is wrong.

The recovery cell replaces all patient rows with the verified clone's rows. Compare the next result with your saved original city counts. If the clone is missing or uncertain, do not guess: follow the guide's exact-data recovery path, which resets all three lab tables. If `CLINICAL_NOTES` is still dropped, run the UNDROP cell before any whole-dataset reset.

After recovery, you can repeat the lesson from the timestamp cell, using the verified clone. Do not recreate or replace the safety copy.

In [ ]:
-- RECOVERY ONLY: facilitator must verify the unchanged safety clone first.
INSERT OVERWRITE INTO DAY0_LAB.HEALTHCARE.PATIENTS
SELECT * FROM DAY0_LAB.HEALTHCARE.PATIENTS_BEFORE_EXERCISE;

In [ ]:
SELECT CITY, COUNT(*) AS PATIENTS
FROM DAY0_LAB.HEALTHCARE.PATIENTS
GROUP BY CITY
ORDER BY CITY;

## Finish and stop notebook compute

After both tables are restored, use **Connected**, hover over your notebook service, and select **Suspend** (the pause icon). This clears session variables, so do it only after recovery is complete. Closing the browser alone does not stop notebook compute. Do not suspend a service if another notebook you need is still running on it.

Return to the guide's Streamlit module. Keep this notebook and `explore.sql` with your workshop files. Download edited files before account access expires; notebook outputs are not stored in the `.ipynb` file, so use **Export as HTML** if you want to keep the displayed results.

### Try it with CoCo (optional)

Explain a few practical examples of how I might use Snowflake Time Travel in my role as a financial planning and reporting professional for a health insurance payer. For each example, describe the business problem and how Time Travel could help. Use plain language and fictional examples. Explain the retention limits and why Time Travel is not a long-term reporting archive. Do not query or change any data.

**Success check:** Choose one example relevant to your work and explain how Time Travel could help, including whether the needed history would still be within retention.